In [ ]:
#Google Colab setup for unsloth

%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm
else:
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
    !uv pip install -qqq --no-deps --upgrade "torchao>=0.16.0"
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

In [ ]:
#Lightning.ai setup for unsloth

%pip uninstall -y unsloth unsloth_zoo trl transformers
%pip install --no-cache-dir "transformers==4.56.2" "trl==0.22.2"
%pip install --upgrade --force-reinstall --no-cache-dir --no-deps unsloth unsloth_zoo

In [2]:
# For Windows locally
import os, sys

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_DATASETS_MULTITHREADING_MAX_WORKERS"] = "1"

if sys.platform == "win32":
    from datasets import Dataset

    if not hasattr(Dataset, "_original_map_windows_fix"):
        Dataset._original_map_windows_fix = Dataset.map

        def _win32_safe_map(self, *args, **kwargs):
            kwargs["num_proc"] = None   # force single-process map
            return Dataset._original_map_windows_fix(self, *args, **kwargs)

        Dataset.map = _win32_safe_map

In [3]:
from unsloth import FastLanguageModel
import torch

import os
HF_USERNAME = os.getenv('HF_USERNAME', 'RealPirate786')
HF_TOKEN = os.getenv('HF_TOKEN')


In [ ]:
max_seq_length = 2048
lora_rank = 32
model_path = "Qwen/Qwen3-0.6B"  

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path,
    max_seq_length = max_seq_length,
    load_in_4bit = True, # False for LoRA 16bit
    fast_inference = False, # Enable vllm fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.65, # Reduce if out of memory
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank * 2, # *2 speeds up training
    use_gradient_checkpointing = "unsloth", # Reduces memory usage
    random_state = 42,
)

#### Define System Prompt

In [4]:
# @title
def get_system_prompt():
    
  system_prompt = '''# Minesweeper AI System Prompt

    You are an AI agent playing Minesweeper.
    Your objective is to maximize score while completing the game without revealing a mine.

    ## Allowed Actions

    You send exactly one move at a time using one of these actions:

    - `reveal`
    - `flag`

    Each move must target exactly one tile coordinate:

    - `x`
    - `y`

    Example move payload:

    ```json
    {
      "action": "reveal|flag",
      "x": 3,
      "y": 4
    }
    ```

    ## Rules You Must Follow

    - Revealing a flagged tile is invalid
    - Flagging a revealed tile is invalid
    - Revealing a mine ends the game immediately
    - The game is won only when all safe tiles are revealed and all mines are correctly flagged
    - Hidden mine locations are not exposed while the game is in progress

    ## Scoring Rules

    - RL reward for revealing a numbered tile: `+5`
    - RL reward for revealing a zero tile: `+2`
    - RL reward for correctly flagging a mine: `+10`
    - RL penalty for flagging a safe tile: `-10`
    - RL penalty for revealing a mine: `-25`
    - RL bonus for a forced move: `+3`
    - RL bonus for a frontier move near revealed numbers: `+1`
    - RL penalty for skipping a forced move: `-4`
    - RL penalty for contradicting visible constraints: `-6`
    - RL win bonus: `+50`
    - RL rewards are separate from the gameplay scoring system

    ## State Interpretation

    You receive game state that includes:
    - `board`

    Symbol meanings:

    - `.` means the tile is unrevealed
    - `F` means the tile is flagged
    - `_` means the tile is revealed and has zero adjacent mines
    - `"1"` to `"8"` mean the tile is revealed and the value is the number of adjacent mines
    - `B` means a bomb tile visible after the game is lost

    Example:

    ```text
    . . . 1
    _ F 1 .
    _ 2 3 _
    _ _ _ _
    ```

    Interpret the array using zero-based coordinates:

    - `x` is the column index
    - `y` is the row index
    - `board[y][x]` is the tile value

    Tile state meanings:

    - `hidden`: unrevealed and unflagged
    - `flagged`: currently flagged as a mine candidate
    - `revealed`: safely revealed
    - `mine`: appears only in terminal loss state

    Interpretation of `adjacent_mines`:

    - If `revealed`, the value is the number of adjacent mines
    - If `0`, the tile has no adjacent mines
    - If hidden or flagged during active play, `adjacent_mines` may be `null`

    ## Strategy Guidance

    - Prefer moves that are logically certain
    - Use revealed numbers to infer safe tiles and mine tiles
    - Flag tiles only when there is strong justification, because incorrect flags lose points
    - Prefer guaranteed safe reveals over speculative flags when uncertainty is high
    - Use the safe first move to open information quickly
    - Track local constraints around numbered tiles
    - Avoid random reveals unless no deterministic move exists
    - If forced to guess, choose the move with the lowest estimated mine risk
    - Output only required json

    ## Decision Policy

    For a given board state:
    1. Read the full visible board state
    2. interpret `board[y][x]` using the compact symbol rules
    3. Identify deterministic safe reveals
    4. Identify deterministic mine flags
    5. If no deterministic move exists, estimate the least risky hidden tile
    6. Return exactly one move
    '''
  return system_prompt


system_prompt = get_system_prompt()

#### Define Custom Chat Template

In [5]:
def get_chat_template():
    chat_template = '''
    {%- set eos = eos_token if eos_token is defined and eos_token is string and eos_token != '<|im_end|>' else '' -%}

    {%- if messages[0].role == 'system' %}
        {{- '<|im_start|>system\\n' + messages[0].content + '<|im_end|>\\n' }}
    {%- endif %}

    {%- for message in messages %}
        {%- if message.content is string %}
            {%- set content = message.content %}
        {%- else %}
            {%- set content = '' %}
        {%- endif %}

        {%- if message.role == "user" %}
            {{- '<|im_start|>user\\n' + content + '<|im_end|>\\n' }}

        {%- elif message.role == "system" and not loop.first %}
            {{- '<|im_start|>system\\n' + content + '<|im_end|>\\n' }}

        {%- elif message.role == "assistant" %}
            {%- set reasoning_content = '' %}

            {%- if message.reasoning_content is string %}
                {%- set reasoning_content = message.reasoning_content %}
            {%- else %}
                {%- if '</think>' in content %}
                    {%- set reasoning_content = content.split('</think>')[0].rstrip('\\n').split('<think>')[-1].lstrip('\\n') %}
                    {%- set content = content.split('</think>')[-1].lstrip('\\n') %}
                {%- endif %}
            {%- endif %}

            {%- if reasoning_content %}
                {{- '<|im_start|>assistant\\n<think>\\n' + reasoning_content.strip('\\n') + '\\n</think>\\n\\n' + content.lstrip('\\n') + '<|im_end|>' + eos + '\\n' }}
            {%- else %}
                {{- '<|im_start|>assistant\\n' + content + '<|im_end|>' + eos + '\\n' }}
            {%- endif %}
        {%- endif %}
    {%- endfor %}

    {%- if add_generation_prompt %}
        {{- '<|im_start|>assistant\\n' }}
    {%- endif %}
    '''
    return chat_template

chat_template = get_chat_template()

In [ ]:
tokenizer.chat_template = chat_template

#### Load The Datasets for SFT

In [6]:
import pandas as pd
train = pd.read_csv('D:\Projects\Minesweeper LLM\dataset\dataset_train_9x9.csv')
test = pd.read_csv('D:\Projects\Minesweeper LLM\dataset\dataset_test_9x9.csv')

Chat Template Example

In [7]:
user = f"Board State: {train['input'][0]}\nMax Mines: {train['max_mines'][0]}\nMax Rows: {train['rows'][0]}\nMax Columns: {train['columns'][0]}"
assistant = train['output'][0]
tokenizer.apply_chat_template(
    [
        {"role": "system", "content": get_system_prompt()},
        {"role": "user", "content": user},
        {"role": "assistant", "content": assistant},
    ],
    tokenize = False,
    add_generation_prompt = False,
)



'<|im_start|>system\n# Minesweeper AI System Prompt\n\n    You are an AI agent playing Minesweeper.\n    Your objective is to maximize score while completing the game without revealing a mine.\n\n    ## Allowed Actions\n\n    You send exactly one move at a time using one of these actions:\n\n    - `reveal`\n    - `flag`\n\n    Each move must target exactly one tile coordinate:\n\n    - `x`\n    - `y`\n\n    Example move payload:\n\n    ```json\n    {\n      "action": "reveal|flag",\n      "x": 3,\n      "y": 4\n    }\n    ```\n\n    ## Rules You Must Follow\n\n    - Revealing a flagged tile is invalid\n    - Flagging a revealed tile is invalid\n    - Revealing a mine ends the game immediately\n    - The game is won only when all safe tiles are revealed and all mines are correctly flagged\n    - Hidden mine locations are not exposed while the game is in progress\n\n    ## Scoring Rules\n\n    - RL reward for revealing a numbered tile: `+5`\n    - RL reward for revealing a zero tile: `+2

#### Format the dataset

In [8]:
def format_dataset_train(df):
    input = f"Board State: {df['input']}\nMax Mines: {df['max_mines']}\nMax Rows: {df['rows']}\nMax Columns: {df['columns']}"
    output = df['output']

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": input},
        {"role": "assistant", "content": output},
    ]

def format_dataset_test(df):
    input = f"Board State: {df['input']}\nMax Mines: {df['max_mines']}\nMax Rows: {df['rows']}\nMax Columns: {df['columns']}"
    output = df['output']

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": input},
    ]

train['Messages'] = train.apply(format_dataset_train, axis=1)
test['Messages'] = test.apply(format_dataset_test, axis=1)

#check the formatted messages
test['Messages'][0]

[{'role': 'system',
  'content': '# Minesweeper AI System Prompt\n\n    You are an AI agent playing Minesweeper.\n    Your objective is to maximize score while completing the game without revealing a mine.\n\n    ## Allowed Actions\n\n    You send exactly one move at a time using one of these actions:\n\n    - `reveal`\n    - `flag`\n\n    Each move must target exactly one tile coordinate:\n\n    - `x`\n    - `y`\n\n    Example move payload:\n\n    ```json\n    {\n      "action": "reveal|flag",\n      "x": 3,\n      "y": 4\n    }\n    ```\n\n    ## Rules You Must Follow\n\n    - Revealing a flagged tile is invalid\n    - Flagging a revealed tile is invalid\n    - Revealing a mine ends the game immediately\n    - The game is won only when all safe tiles are revealed and all mines are correctly flagged\n    - Hidden mine locations are not exposed while the game is in progress\n\n    ## Scoring Rules\n\n    - RL reward for revealing a numbered tile: `+5`\n    - RL reward for revealing a z

#### Convert to HuggingFace compatible dataset

In [9]:
from datasets import Dataset
MAX_TRAIN_EXAMPLES = 1400
MAX_TEST_EXAMPLES = 600
train_small_batch = train.iloc[:MAX_TRAIN_EXAMPLES]
test_small_batch = test.iloc[:MAX_TEST_EXAMPLES]

train_small_batch["text"] = tokenizer.apply_chat_template(train_small_batch["Messages"].values.tolist(), tokenize = False, add_generation_prompt = False)
test_small_batch["text"] = tokenizer.apply_chat_template(test_small_batch["Messages"].values.tolist(), tokenize = False, add_generation_prompt = False)

train_small_batch = Dataset.from_pandas(train_small_batch)
test_small_batch = Dataset.from_pandas(test_small_batch)

Delete the datasets that are not needed from memory

In [10]:
del train
del test
import gc
torch.cuda.empty_cache()
gc.collect()

32

#### Start the Finetuning-Process

In [11]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_small_batch,
    args = SFTConfig(
        dataset_text_field = "text",
        dataset_num_proc =1,
        dataloader_num_workers=0,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 1, # Use GA to mimic batch size!
        warmup_steps = 5,
        num_train_epochs = 2, # Set this for 1 full training run.
        learning_rate = 2e-5, # Reduce to 2e-5 for long training runs
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 42,
        report_to = "wandb", # Use TrackIO/WandB etc
    ),
)

Unsloth: Tokenizing ["text"]:   0%|          | 0/1400 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [12]:
batch = next(iter(trainer.get_train_dataloader()))

input_ids = batch["input_ids"][0].cpu()
labels = batch["labels"][0].cpu()

for i in range(len(input_ids) - 80, len(input_ids)):
    tid = int(input_ids[i])
    lid = int(labels[i])

    token = tokenizer.decode([tid], skip_special_tokens=False)

    if lid == -100:
        print(i, "MASKED ", repr(token))
    else:
        print(i, "TRAINED", repr(token))

1024 TRAINED "',"
1025 TRAINED " '"
1026 TRAINED '1'
1027 TRAINED "',"
1028 TRAINED " '"
1029 TRAINED '2'
1030 TRAINED "',"
1031 TRAINED " '"
1032 TRAINED '2'
1033 TRAINED "',"
1034 TRAINED " '"
1035 TRAINED '3'
1036 TRAINED "',"
1037 TRAINED " '"
1038 TRAINED 'F'
1039 TRAINED "',"
1040 TRAINED " '"
1041 TRAINED '4'
1042 TRAINED "',"
1043 TRAINED " '"
1044 TRAINED 'F'
1045 TRAINED "'],"
1046 TRAINED " ['"
1047 TRAINED ".',"
1048 TRAINED " '"
1049 TRAINED '1'
1050 TRAINED "',"
1051 TRAINED " '.',"
1052 TRAINED " '.',"
1053 TRAINED " '"
1054 TRAINED '1'
1055 TRAINED "',"
1056 TRAINED " '.',"
1057 TRAINED " '.',"
1058 TRAINED " '.',"
1059 TRAINED " '."
1060 TRAINED "']]\n"
1061 TRAINED 'Max'
1062 TRAINED ' Mines'
1063 TRAINED ':'
1064 TRAINED ' '
1065 TRAINED '2'
1066 TRAINED '4'
1067 TRAINED '\n'
1068 TRAINED 'Max'
1069 TRAINED ' Rows'
1070 TRAINED ':'
1071 TRAINED ' '
1072 TRAINED '9'
1073 TRAINED '\n'
1074 TRAINED 'Max'
1075 TRAINED ' Columns'
1076 TRAINED ':'
1077 TRAINED ' '
1078 TRA

Run the trainer

In [13]:
try:
    trainer.train()
except KeyboardInterrupt:
    print("Training interrupted.")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,400 | Num Epochs = 2 | Total steps = 2,800
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 1 x 1) = 1
 "-____-"     Trainable parameters = 20,185,088 of 616,235,008 (3.28% trained)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Kartikeya Srivastava\_netrc.
wandb: Currently logged in as: kartikeyasrivastava769 (kartikeyasrivastava769-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
5,2.250814
10,0.968287
15,0.298023
20,0.155194
25,0.098794


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Training interrupted.


wandb: WARNING Tried to log to step 1 that is less than the current step 15. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 2 that is less than the current step 25. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 3 that is less than the current step 35. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 4 that is less than the current step 45. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 5 that is less than the current step 55. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/

#### Test the trained model

In [14]:
from transformers import TextStreamer
FastLanguageModel.for_inference(model)

im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")

stop_ids = [tokenizer.eos_token_id]

if (
    im_end_id is not None
    and im_end_id != tokenizer.unk_token_id
    and im_end_id not in stop_ids
):
    stop_ids.append(im_end_id)

streamer = TextStreamer(
    tokenizer,
    skip_prompt=True,
    skip_special_tokens=True,
)

text = tokenizer.apply_chat_template(
    test_small_batch["Messages"][50],
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
)
inputs = tokenizer(text, return_tensors="pt").to("cuda")

_ = model.generate(
    **inputs,
    max_new_tokens=32,
    do_sample=False,
    eos_token_id=stop_ids,
    pad_token_id=tokenizer.eos_token_id,
    streamer=streamer,
)
print('---')

d:\Projects\Minesweeper LLM\.venv\Lib\site-packages\transformers\modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
d:\Projects\Minesweeper LLM\.venv\Lib\site-packages\transformers\modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


{"action": "flag", "x": 3, "y": 4}
---


#### Save the model

Save as .safetensors 16-bit

In [ ]:
# Save the model to Hugging Face Hub. You can load this model later for further training using GRPO or for inference.
def save_model():
    model.save_pretrained_merged("Minesweeper_agent_Qwen3_0.6B", tokenizer, save_method = "merged_16bit",)
    model.push_to_hub_merged(f"{HF_USERNAME}/Minesweeper_agent_Qwen3_0.6B-SFT", tokenizer, save_method = "merged_16bit", token = HF_TOKEN)

save_model() #Uncomment this to save model

Save as GGUF format

In [ ]:
# Save the model in GGUF format for using it through llama.cpp
def save_model_gguf():
    model.save_pretrained_gguf("Minesweeper_agent_Qwen3_4B_2507", tokenizer,)
    model.push_to_hub_gguf(f"{HF_USERNAME}/Minesweeper_agent_Qwen3_4B_2507-GGUF", tokenizer, token = HF_TOKEN, quantization_method = 'q8_0')

# save_model_gguf() #Uncomment this to save model

### Reinforcement Learning with GRPO

In [1]:
#Use this if you want to use your own pre-trained model. Useful when you save previous model first first and then want to load it for further training using GRPO. No need to use this if you are training model directly after SFT.
from unsloth import FastLanguageModel
max_seq_length = 2048
lora_rank = 32
model_path = r"D:\Projects\Minesweeper LLM\notebook\Minesweeper_agent_Qwen3_0.6B"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path,
    max_seq_length = max_seq_length,
    load_in_4bit = True, # False for LoRA 16bit
    fast_inference = False, # Enable vllm fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.75, # Reduce if out of memory
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank * 2, # *2 speeds up training
    use_gradient_checkpointing = "unsloth", # Reduces memory usage
    random_state = 42,
)

model.generation_config.max_length = None


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0703 17:32:02.815000 22200 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Qwen3 patching. Transformers: 5.12.1. vLLM: 0.24.0+cu132.
   \\   /|    NVIDIA GeForce RTX 3060 Laptop GPU. Num GPUs = 1. Max memory: 6.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.6. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

d:\Projects\Minesweeper LLM\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
The tokenizer you are loading from 'D:\Projects\Minesweeper LLM\notebook\Minesweeper_agent_Qwen3_0.6B' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from 'D:\Projects\Minesweeper LLM\notebook\Minesweeper_agent_Qwen3_0.6B' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading th

In [15]:
tokenizer.chat_template = get_chat_template()

#### Define Reward Function


In [16]:
import random
import sys
from pathlib import Path
from datasets import Dataset

base_dir = Path.cwd().parent
print("Notebook cwd:", base_dir)

project_dir = base_dir / "src"
print("Project dir:", project_dir)

if not project_dir.exists():
    raise FileNotFoundError(f"Folder not found: {project_dir}")

# Add the project folder to Python path
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

print("Added to sys.path:", sys.path[0])

from minesweeper.engine import GameConfig, GameEngine

#Create the game first
def sample_prefill_move(game: GameEngine, rng: random.Random, reveal_probability=0.7):
    reveal_candidates = [tile for tile in game.iter_tiles() if not tile.is_revealed and not tile.is_flagged]
    flag_candidates = [tile for tile in game.iter_tiles() if not tile.is_revealed and not tile.is_flagged]

    if not reveal_candidates and not flag_candidates:
        return None

    if reveal_candidates and (not flag_candidates or rng.random() < reveal_probability):
        tile = rng.choice(reveal_candidates)
        return {"action": "reveal", "x": tile.x, "y": tile.y}

    tile = rng.choice(flag_candidates)
    return {"action": "flag", "x": tile.x, "y": tile.y}


def create_game(width=9, height=9, mine_density=0.15, seed=None, output_format="compact", max_prefill_moves=4):
    rng = random.Random(seed) if seed is not None else random.Random()
    game = GameEngine(config=GameConfig(width=width, height=height, mine_density=mine_density), rng=rng)

    prefill_moves = rng.randint(0, max_prefill_moves)

    if prefill_moves > 0:
        first_x = rng.randrange(width)
        first_y = rng.randrange(height)
        game.reveal(first_x, first_y)

        for _ in range(prefill_moves - 1):
            if game.status.value != "in_progress":
                break

            move = sample_prefill_move(game, rng)
            if move is None:
                break

            try:
                if move["action"] == "reveal":
                    game.reveal(move["x"], move["y"])
                else:
                    game.flag(move["x"], move["y"])
            except ValueError:
                continue

    visible_state = game.compact_state()
    full_board_state = game.full_board_compact_state() if game.snapshot()["mines_placed"] else None
    return [visible_state, full_board_state, game.snapshot()]

#create a batch of games with random configurations
def create_game_batch(num_games = 5):
    board_size = [(5, 5), (9, 9), (12, 12), (15, 15), (15, 20), (12, 15), (20, 20)]
    mine_density = [0.15, 0.30]
    
    games = []
    for _ in range(num_games):
        width, height = random.choice(board_size)
        density = random.choice(mine_density)
        game = create_game(width=width, height=height, mine_density=density)
        games.append(game)
    
    return games

#Helper function to build user prompt from game state
def build_user_prompt_from_state(game):
    user_prompt = f"Board State: {game['board']}\nMax Mines: {game['mine_count']}\nMax Rows: {game['height']}\nMax Columns: {game['width']}"
    return user_prompt


def create_dataset_from_games(games):
    rows = []
    # Set to None to use raw user prompts without chat template formatting
    for game in games:
        user_prompt = build_user_prompt_from_state(game[0])
        rows.append(
            {
                "prompt": tokenizer.apply_chat_template(
                    [
                        {"role": "system", "content": get_system_prompt()},
                        {"role": "user", "content": user_prompt}
                    ],
                    tokenize=False,
                    add_generation_prompt=True
                ) ,
                "revealed_board": str(game[1]),
                "snapshot": str(game[2]),
            }
        )
    return Dataset.from_list(rows)


def build_live_dataset(num_examples, board_sizes, mine_densities, seed=None):
  rng = random.Random(seed)
  rows = []

  while len(rows) < num_examples:
      width, height = rng.choice(board_sizes)
      density = rng.choice(mine_densities)

      game = create_game(width=width, height=height, mine_density=density)
      visible_state, full_state, snapshot = game[0], game[1], game[2]

      if visible_state["status"] != "in_progress":
          continue

      user_prompt = build_user_prompt_from_state(visible_state)

      rows.append({
          "prompt": tokenizer.apply_chat_template(
              [
                  {"role": "system", "content": get_system_prompt()},
                  {"role": "user", "content": user_prompt},
              ],
              tokenize=False,
              add_generation_prompt=True,
          ),
          "rows": visible_state["height"],
          "columns": visible_state["width"],
          "max_mines": visible_state["mine_count"],
          "board_state": str(visible_state["board"]),
          "revealed_board": str(full_state),
          "snapshot": str(snapshot),
      })

  return Dataset.from_list(rows)    # str(f"Board State: {train['input'][0]}\nMax Mines: {train['max_mines'][0]}\nMax Rows: {train['rows'][0]}\nMax Columns {train['columns'][0]}")


# game = create_game()
# print(game)  

    # str(f"Board State: {train['input'][0]}\nMax Mines: {train['max_mines'][0]}\nMax Rows: {train['rows'][0]}\nMax Columns {train['columns'][0]}")


Notebook cwd: d:\Projects\Minesweeper LLM
Project dir: d:\Projects\Minesweeper LLM\src
Added to sys.path: d:\Projects\Minesweeper LLM\src
pygame 2.6.1 (SDL 2.28.4, Python 3.12.12)
Hello from the pygame community. https://www.pygame.org/contribute.html


d:\Projects\Minesweeper LLM\.venv\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [17]:
import ast
import json
import re
from minesweeper.rl_rewards import calculate_reward_components

rewards = []

MALFORMED_COMPLETION_PENALTY = -1.0
SNAPSHOT_LOAD_PENALTY = -1.0
INVALID_MOVE_PENALTY = -0.8
EXECUTION_ERROR_PENALTY = -0.66

def build_game_from_state(snapshot):
    game = GameEngine.from_snapshot(ast.literal_eval(snapshot))
    return game

def parse_completion_payload(completion):
    return json.loads(completion.strip())

def validate_move(rows, columns, response):
    action = response.get("action")
    x = response.get("x")
    y = response.get("y")

    if action not in {"reveal", "flag"}:
        return False
    if not isinstance(x, int) or not isinstance(y, int):
        return False
    if x < 0 or x >= columns or y < 0 or y >= rows:
        return False
    return True

def get_reward_context(i, rows_list, columns_list, snapshots):
    rows = rows_list[i] if i < len(rows_list) else None
    columns = columns_list[i] if i < len(columns_list) else None
    snapshot = snapshots[i] if i < len(snapshots) else None
    if snapshot is None or rows is None or columns is None:
        raise ValueError(f"Missing required snapshot information for reward calculation at index {i}")
    return rows, columns, snapshot

def score_move_reward(game, response):
    return calculate_reward_components(game, response)

def score_completion_reward(i, completion, rows_list, columns_list, snapshots):
    rows, columns, snapshot = get_reward_context(i, rows_list, columns_list, snapshots)

    try:
        response = parse_completion_payload(completion)
    except Exception:
        return {
            "reward": float(MALFORMED_COMPLETION_PENALTY),
            "component": "malformed_completion",
        }

    try:
        game = build_game_from_state(snapshot)
    except Exception:
        return {
            "reward": float(SNAPSHOT_LOAD_PENALTY),
            "component": "snapshot_load_failure",
        }

    if not validate_move(rows, columns, response):
        return {
            "reward": float(INVALID_MOVE_PENALTY),
            "component": "invalid_move",
        }

    try:
        reward_components = score_move_reward(game, response)
    except ValueError as e:
        error_string = str(e)
        if "Revealed tiles cannot be flagged." in error_string:
            return {
                "reward": float(-0.5),
                "component": "revealed tiles cannot be flagged",
            }
                
        return {
            "reward": float(EXECUTION_ERROR_PENALTY),
            "component": "execution_error",
        }

    return {
        "reward": float(reward_components["total_reward"]),
        "component": "move_reward",
        "reward_components": reward_components,
    }

def calculate_reward(prompts=None, completions=None, **kwargs):
    rows_list = kwargs.get("rows") or []
    columns_list = kwargs.get("columns") or []
    snapshots = kwargs.get("snapshot") or []

    rewards = []
    reward_logs = []

    for i, completion in enumerate(completions or []):
        reward_result = score_completion_reward(i, completion, rows_list, columns_list, snapshots)
        rewards.append(reward_result["reward"])
        reward_logs.append(reward_result)

    calculate_reward.last_logs = reward_logs
    return rewards

#### Set GRPO config and sampling parameters

In [18]:
from trl import GRPOConfig, GRPOTrainer
from minesweeper.wandb_logging import WandbRewardLoggerCallback
from vllm import SamplingParams
vllm_sampling_params = SamplingParams(
    top_p=0.95,
    top_k=50,
    seed=42,
    temperature = 1.5,
    stop=[tokenizer.eos_token],
    include_stop_str_in_output=True,
)

training_args = GRPOConfig(
    vllm_sampling_params=vllm_sampling_params,
    temperature=1.0,
    learning_rate=5e-6,
    weight_decay=0.001,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    optim="adamw_8bit",
    logging_steps=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=6,
    num_generations=6,
    max_prompt_length=max_seq_length - 32,
    max_completion_length=32,
    max_steps=400,
    save_steps=40,
    report_to="wandb",
    run_name="minesweeper-grpo-debug",
    output_dir="outputs",
    max_grad_norm = 1.0
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [19]:
# GRPO trainer using live engine states instead of the offline CSV dataset
board_sizes = [(9, 9)]
mine_densities = [0.15, 0.30]
live_train_dataset = build_live_dataset(
    num_examples=300,
    board_sizes=board_sizes,
    mine_densities=mine_densities,
    seed=42
)

grpo_trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[calculate_reward],
    args=training_args,
    train_dataset=live_train_dataset,
)

grpo_trainer.add_callback(WandbRewardLoggerCallback(calculate_reward))
grpo_trainer.train()


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 300 | Num Epochs = 2 | Total steps = 400
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 6
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 6 x 1) = 6
 "-____-"     Trainable parameters = 20,185,088 of 616,235,008 (3.28% trained)
Passing `generation_config` together with generation-related arguments=({'cache_implementation', 'disable_compile', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
d:\Projects\Minesweeper LLM\.venv\Lib\site-packages\transformers\modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECAT

Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / calculate_reward / mean,rewards / calculate_reward / std
1,0.001272,-0.775000,0.551135,29.833334,19.000000,32.000000,0.833333,19.000000,19.000000,19.000000,1.271922,-0.775000,0.551135
2,0.003559,-1.000000,0.000000,23.666668,1.000000,32.000000,0.500000,15.333334,1.000000,23.000000,3.559039,-1.000000,0.000000
3,0.000841,-1.000000,0.000000,22.000000,2.000000,32.000000,0.666667,2.000000,2.000000,2.000000,0.840925,-1.000000,0.000000
4,0.001128,-1.000000,0.000000,32.000000,32.000000,32.000000,1.000000,0.000000,0.000000,0.000000,1.128490,-1.000000,0.000000
5,0.001021,-0.341667,0.722092,25.500000,19.000000,32.000000,0.500000,19.000000,19.000000,19.000000,1.020856,-0.341667,0.722092
6,0.001232,-1.000000,0.000000,14.333334,2.000000,32.000000,0.333333,5.500000,2.000000,16.000000,1.231610,-1.000000,0.000000
7,0.001703,-0.775000,0.551135,19.833334,2.000000,32.000000,0.333333,13.750000,2.000000,25.000000,1.702679,-0.775000,0.551135
8,0.001638,-1.000000,0.000000,28.500000,11.000000,32.000000,0.666667,21.500000,11.000000,32.000000,1.638202,-1.000000,0.000000
9,0.001969,-1.000000,0.000000,19.666668,1.000000,32.000000,0.333333,13.500000,1.000000,30.000000,1.968632,-1.000000,0.000000
10,0.001286,-1.000000,0.000000,29.833334,19.000000,32.000000,0.833333,19.000000,19.000000,19.000000,1.286314,-1.000000,0.000000


Unsloth: Restored added_tokens_decoder metadata in outputs\checkpoint-40\tokenizer_config.json.
d:\Projects\Minesweeper LLM\.venv\Lib\site-packages\transformers\modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
d:\Projects\Minesweeper LLM\.venv\Lib\site-packages\transformers\modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Unsloth: Restored added_tokens_decoder metadata in outputs\checkpoint-80\tokenizer_config.json.
d:\Projects\Minesweeper LLM\.venv\Lib\site-packages\transform

TrainOutput(global_step=400, training_loss=0.0008595971670001746, metrics={'train_runtime': 1849.0562, 'train_samples_per_second': 1.298, 'train_steps_per_second': 0.216, 'total_flos': 0.0, 'train_loss': 0.0008595971670001746})

Start reinforcement training of the model using GRPO

In [ ]:
grpo_trainer.train()

#### Save the GRPO trained model

In [ ]:
save_model() 
save_model_gguf()

#### Play a full game with the learned model

In [23]:
import json
import random

from minesweeper.engine import GameConfig, GameEngine
from unsloth import FastLanguageModel


def render_compact_board(board):
    return "\n".join(" ".join(str(cell) for cell in row) for row in board)


def build_live_user_prompt(state):
    return (
        f"Board State: {state['board']}\n"
        f"Score: {state['score']}\n"
        f"Max Mines: {state['mine_count']}\n"
        f"Max Rows: {state['height']}\n"
        f"Max Columns: {state['width']}"
    )


def generate_model_move(state, *, max_new_tokens=32):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": build_live_user_prompt(state)},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    stop_ids = [tokenizer.eos_token_id]
    im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
    if (
        im_end_id is not None
        and im_end_id != tokenizer.unk_token_id
        and im_end_id not in stop_ids
    ):
        stop_ids.append(im_end_id)

    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        eos_token_id=stop_ids,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=False).strip()

    for stop_token in ("<|im_end|>", "<|endoftext|>"):
        if stop_token in generated_text:
            generated_text = generated_text.split(stop_token, 1)[0].strip()

    move = parse_completion_payload(generated_text)
    return generated_text, move


def play_full_game_with_model(width=9, height=9, mine_density=0.15, seed=42, max_turns=200):
    FastLanguageModel.for_inference(model)
    game = GameEngine(
        config=GameConfig(width=width, height=height, mine_density=mine_density),
        rng=random.Random(seed),
    )

    turn = 1
    while game.status.value == "in_progress" and turn <= max_turns:
        state = game.compact_state()
        print(f"Turn {turn}")
        print(render_compact_board(state["board"]))
        print(f"Score: {state['score']} | Status: {state['status']} | Flags: {state['flagged_count']}/{state['mine_count']}")

        try:
            raw_move, move = generate_model_move(state)
            print(f"Model move: {raw_move}")
            if move["action"] == "reveal":
                result = game.reveal(move["x"], move["y"])
            else:
                result = game.flag(move["x"], move["y"])
            print(f"Result: {result.message} | Move score delta: {result.score_delta:+d}")
        except Exception as exc:
            print(f"Model move failed: {exc}")
            break

        updated_state = game.compact_state()
        print(render_compact_board(updated_state["board"]))
        print(f"Score: {updated_state['score']} | Status: {updated_state['status']} | Flags: {updated_state['flagged_count']}/{updated_state['mine_count']}")
        print("-" * 60)
        turn += 1

    final_state = game.compact_state()
    print("Final board state:")
    print(render_compact_board(final_state["board"]))
    print(f"Final score: {final_state['score']} | Final status: {final_state['status']}")
    return game


# Example run
played_game = play_full_game_with_model(width=9, height=9, mine_density=0.15, seed=42)


Turn 1
. . . . . . . . .
. . . . . . . . .
. . . . . . . . .
. . . . . . . . .
. . . . . . . . .
. . . . . . . . .
. . . . . . . . .
. . . . . . . . .
. . . . . . . . .
Score: 0 | Status: in_progress | Flags: 0/12
Model move: {"action": "reveal", "x": 5, "y": 5}
Result: Reveal processed. | Move score delta: +82
. . . . . . . . .
. . . . . . . . .
. . . . . 3 1 2 .
. . . . . 1 0 1 .
. . 1 1 1 1 0 1 1
. . 1 0 0 0 0 0 0
. . 1 0 0 0 1 1 1
1 1 1 0 0 0 2 . .
0 0 0 0 0 0 2 . .
Score: 82 | Status: in_progress | Flags: 0/12
------------------------------------------------------------
Turn 2
. . . . . . . . .
. . . . . . . . .
. . . . . 3 1 2 .
. . . . . 1 0 1 .
. . 1 1 1 1 0 1 1
. . 1 0 0 0 0 0 0
. . 1 0 0 0 1 1 1
1 1 1 0 0 0 2 . .
0 0 0 0 0 0 2 . .
Score: 82 | Status: in_progress | Flags: 0/12
Model move: {"action": "reveal", "x": 1, "y": 6}
Result: Mine revealed. Game over. | Move score delta: +0
. . . B B . . . .
. . B . B B . . B
. . . . . 3 1 2 .
. B . . B 1 0 1 B
. . 1 1 1 1 0 1 1
. . 1 0